In [1]:
import requests
import pandas as pd
import sqlite3

def extrair_url_escudo(escudos: dict) -> str:
    """Extrai a URL do escudo 60x60 do dicionário aninhado do Cartola."""
    if isinstance(escudos, dict):
        return escudos.get('60x60')
    return None

def limpar_nome_time(linha: pd.Series) -> str:
    """Define o nome oficial do time usando um De-Para para corrigir acentuações e siglas."""
    
    # O nosso dicionário de correções (De-Para)
    # Aqui colocamos os slugs problemáticos da Série A e como eles devem aparecer na tela.
    correcoes = {
        'atletico-mg': 'Atlético-MG',
        'atletico-pr': 'Athletico-PR', # Atualizamos para a grafia nova com H!
        'atletico-go': 'Atlético-GO',
        'sao-paulo': 'São Paulo',
        'gremio': 'Grêmio',
        'corinthians': 'Corinthians',
        'vitoria': 'Vitória',
        'goias': 'Goiás',
        'ceara': 'Ceará',
        'cuiaba': 'Cuiabá',
        'avai': 'Avaí',
        'parana': 'Paraná'
    }
    
    slug = linha.get('slug')
    nome_fantasia = str(linha.get('nome_fantasia'))
    
    # 1. Se o time está no nosso De-Para, usamos a nossa correção escrita à mão
    if pd.notna(slug) and slug in correcoes:
        return correcoes[slug]
    
    # 2. Se o nome_fantasia for muito curto (ex: "COR", "PAL"), tentamos capitalizar o slug
    if len(nome_fantasia) <= 3 and pd.notna(slug) and str(slug).strip() != "":
        return str(slug).replace('-', ' ').title()
    
    # 3. Caso contrário, o nome_fantasia (que costuma ter acentos, ex: "América-RN") já é o correto
    return nome_fantasia


# =============================================================================
# 1. EXTRAÇÃO
# =============================================================================
url_clubes = "https://api.cartola.globo.com/clubes"
resposta = requests.get(url_clubes)

if resposta.status_code == 200:
    dados_clubes = resposta.json()
    
    # Remove o clube genérico/fake (ID '1') do sistema do Cartola de forma segura
    dados_clubes.pop('1', None)

    # =============================================================================
    # 2. TRANSFORMAÇÃO
    # =============================================================================
    df_times = pd.DataFrame.from_dict(dados_clubes, orient='index')

    # Aplica as regras de negócio de desempacotamento e limpeza
    df_times['url_escudo'] = df_times['escudos'].apply(extrair_url_escudo)
    df_times['nome'] = df_times.apply(limpar_nome_time, axis=1)

    # Filtra apenas as colunas do nosso Modelo Dimensional
    df_times = df_times[['id', 'nome', 'abreviacao', 'url_escudo']]

    display(df_times.head(10))

    # =============================================================================
    # 3. CARGA
    # =============================================================================
    try:
        # O 'with' gerencia a conexão, abrindo e fechando automaticamente
        with sqlite3.connect('banco_brasileirao.db') as conexao:
            df_times.to_sql(name='dim_times', con=conexao, if_exists='replace', index=False)
        print("✅ Nova 'dim_times' de 2026 tratada e carregada com sucesso!")
    except Exception as erro:
        print(f"❌ Erro ao salvar no banco de dados: {erro}")
        
else:
    print(f"❌ Falha na Extração. Status Code: {resposta.status_code}")

,id,nome,abreviacao,url_escudo
1349,1349,Ipatinga,IPA,https://s3.glbimg.com/v1/AUTH_58d78b787ec34892...
1371,1371,Cuiabá,CUI,https://s3.glbimg.com/v1/AUTH_58d78b787ec34892...
1390,1390,Icasa,ICA,https://s3.glbimg.com/v1/AUTH_58d78b787ec34892...
2190,2190,Oeste,OES,https://s3.glbimg.com/v1/AUTH_58d78b787ec34892...
2193,2193,Duque de Caxias,DUQ,https://s3.glbimg.com/v1/AUTH_58d78b787ec34892...
2197,2197,Americana,AME,https://s3.glbimg.com/v1/AUTH_58d78b787ec34892...
2305,2305,Mirassol,MIR,https://s3.glbimg.com/v1/AUTH_58d78b787ec34892...
2554,2554,Grêmio Prudente,PRU,https://s3.glbimg.com/v1/AUTH_58d78b787ec34892...
2565,2565,Luverdense,LUV,https://s3.glbimg.com/v1/AUTH_58d78b787ec34892...
262,262,Flamengo,FLA,https://s3.glbimg.com/v1/AUTH_58d78b787ec34892...


✅ Nova 'dim_times' de 2026 tratada e carregada com sucesso!
